In [ ]:
# ========== 全球市场新闻摘要（Day 1 社区练习）==========
#
# 练习目标（理念）
# - 从 Yahoo Finance 抓取最新全球市场新闻链接与正文
# - 把多篇正文合并后，经 OpenRouter（OpenAI 兼容 API）生成客观、简洁的每日市场简报
# - 简报覆盖：整体情绪、异动板块/公司、投资者要点
#
# 和本课 Day 1 的关系
# | 本课概念              | 本练习里你会看到                          |
# |-----------------------|-------------------------------------------|
# | requests + BeautifulSoup | 抓取新闻列表页与文章正文               |
# | system / user messages | 市场分析师人设 + 合并正文作为用户输入   |
# | Chat Completions API  | OpenRouter 上的 gpt-4o-mini               |
# | Markdown 展示         | display(Markdown(...))                    |
#
# 怎么跑
# 1. 准备 .env：OPENROUTER_API_KEY（通常以 sk-or- 开头）
# 2. 从上到下依次运行单元格（Shift+Enter）
# 3. 注意：抓取依赖外网与页面结构；Yahoo 页面改版可能导致链接过滤失效
#
# Built as a community contribution based on Day 1 of Ed Donner's LLM Engineering course.


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 导入标准库 os：读环境变量（Environment Variables），例如 OpenRouter API Key
import os
# 导入标准库 requests：用 HTTP GET 抓取网页 HTML
import requests
# 导入标准库 time：在连续抓文章之间 sleep，降低被限流风险
import time
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可查询的 DOM 树
from bs4 import BeautifulSoup
# 从 dotenv 导入 load_dotenv：把 .env 文件里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从同目录 scraper 模块导入 fetch_website_contents（后面单元格会再定义同名函数；以你实际运行为准）
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown,display
# 从 openai 导入 OpenAI 客户端类：这里会指向 OpenRouter 的兼容端点
from openai import OpenAI


In [ ]:
# ========== 环境：加载并校验 OpenRouter API Key ==========

# override=True：用 .env 覆盖进程里已有的同名环境变量
load_dotenv(override=True)
# 从环境变量读取 OpenRouter 密钥（名字是 OPENROUTER_API_KEY，不是 OPENAI_API_KEY）
api_key = os.getenv('OPENROUTER_API_KEY')

# 分支检查：缺密钥 / 前缀不对 / 首尾空白 —— 打印英文排错提示（影响排查流程，保留原文）
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-or-"):
    # 注意：判断条件是 sk-or-（OpenRouter），但提示文案仍写 sk-proj-（原作者文案，勿改）
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# ========== 客户端：指向 OpenRouter 的 OpenAI 兼容接口 ==========

# base_url 换成 OpenRouter；api_key 用上一格校验过的 OPENROUTER_API_KEY
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)


In [ ]:
# ========== 抓取正文：模拟浏览器 UA + BeautifulSoup 抽段落 ==========

# User-Agent：假装成常见桌面 Chrome，降低被网站直接拒绝的概率
headers = {'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36'}

def fetch_website_contents(url):
        # GET 目标 URL；headers 带上上面的 UA
        response = requests.get(url, headers=headers)
        # 用 html.parser 解析响应字节流为 soup
        soup = BeautifulSoup(response.content, "html.parser")
        # 取 <title>；没有标题就用占位英文串（影响模型输入，保留原文）
        title = soup.title.string if soup.title else "No title found"
        if soup.body:
            # 删掉脚本/样式/图片/输入框等噪声节点，减少无关文本
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            # 只收集 <p> 段落文本（比整页 get_text 更干净）
            paragraphs = soup.body.find_all("p")
            text = ""
            for p in paragraphs:
                text += p.get_text() + "\n"
        else:
            # 没有 <body> 时正文置空
            text = ""
        # 标题 + 正文，并截断到 2000 字符，控制后续 prompt 体积
        return (title + "\n\n" + text)[:2_000]


In [ ]:
# ========== 抓取链接：收集页面上所有 <a href> ==========

def fetch_website_links(url):
        # GET 列表页 HTML
        response = requests.get(url, headers=headers)
        # 解析为 BeautifulSoup 文档树
        soup = BeautifulSoup(response.content, "html.parser")
        # 找出所有锚点标签
        all_tags = soup.find_all("a")
        links = []
        for link in all_tags:
            # 取 href 属性；可能是相对路径或绝对 URL
            href = link.get("href")
            if href:
                links.append(href)
        return links


In [ ]:
# ========== 取 Yahoo Finance 新闻列表页上的原始链接 ==========

# 固定入口：Yahoo Finance /news/（URL 保留英文，改译会抓错站）
raw_links = fetch_website_links("https://finance.yahoo.com/news/")
# 先打印看看原始 href 长什么样，方便调试过滤规则
print(raw_links)


In [ ]:
# ========== 过滤：只要路径含 articles 且以 .html 结尾的文章链接 ==========

article_links_raw = []

for link in raw_links:
    # 启发式过滤：命中 Yahoo 文章 URL 形态（页面改版时可能要改条件）
    if "articles" in link and link.endswith(".html"):
        article_links_raw.append(link)

# 看过滤后还剩多少条
print(len(article_links_raw))


In [ ]:
# ========== 去重：保持首次出现顺序，去掉重复 href ==========

article_links = []

for link in article_links_raw:
    # 简单 O(n) 去重：列表里还没有才追加
    if link not in article_links:
        article_links.append(link)

# 打印去重后的文章链接列表
print(article_links)


In [ ]:
# ========== 抽样抓正文：只取前 5 篇，篇间 sleep 1 秒 ==========

# 切片前 5 条，控制调用次数与 prompt 长度
selected_links = article_links[:5]

all_articles_content = []

for link in selected_links:
        # 对每篇调用上文定义的 fetch_website_contents
        content = fetch_website_contents(link)
        all_articles_content.append(content)
        # 礼貌等待，减轻对目标站的请求压力
        time.sleep(1)

# 确认抓到几篇，并预览第一篇截断正文
print(len(all_articles_content))
print(all_articles_content[0])


In [ ]:
# ========== 合并多篇正文：用分隔线拼成一段大文本，供 user prompt 使用 ==========

merge_articles = "\n\n---\n\n".join(all_articles_content)


In [ ]:
# ========== Prompt：system 定人设与口吻，user 塞入合并后的新闻正文 ==========

# system prompt 保留英文：这是发给模型的指令，改译会改变回答风格/行为
system_prompt = """
You are a professional market analyst who writes concise, objective daily market wrap-ups.
You will be given the combined content of several news articles about global financial markets.
Summarize them into a clear market briefing.
Do not use humor, sarcasm, or casual language — keep the tone objective and professional, like a Bloomberg or Reuters market wrap.
Respond in markdown. Do not wrap the markdown in a code block.
"""

# user prompt：把 merge_articles 插进 f-string；结构要求也保持英文
user_prompt = f"""
Here is the combined content of several recent global market news articles:

{merge_articles}

Based on these articles, write a market briefing that includes:
1. Overall market sentiment today (bullish/bearish/mixed), with reasoning
2. Sectors or companies that are moving significantly, and why
3. 2-3 key takeaways an investor should know today

Keep it concise and skip anything unrelated to markets (ads, navigation text, unrelated topics).
"""


In [ ]:
# ========== 调用 Chat Completions：非流式一次拿完整简报并 Markdown 展示 ==========

# messages：system + user 两轮角色消息
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# 经 OpenRouter 客户端调用；model id 保持 gpt-4o-mini
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
)

# 从 choices[0].message.content 取出助手文本（变量名复用 response）
response = response.choices[0].message.content

# 在笔记本里用 Markdown 渲染简报
display(Markdown(response))


In [ ]:
# ========== 草稿单元格：原作者未写完的赋值（故意保持残缺，勿「补全」以免改逻辑）==========
response = 
